In [52]:
import pandas as pd
import re
import html
import emoji
from pathlib import Path
from nltk import sent_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer


**Cleaning raw data**

In [53]:
def clean_text(text, verbose=False):
    steps = []

   # Remove HTML tags and decode HTML entities
    text = re.sub(r'<[^>]+>', '', text)
    text = html.unescape(text)
    if verbose:
        steps.append(("1. Remove HTML", text))

    # Remove email addresses
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '', text)
    if verbose:
        steps.append(("2. Remove emails", text))

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    if verbose:
        steps.append(("3. Remove URLs", text))

    # Demojizing emojis
    text = emoji.demojize(text)
    text = text.replace(":", " ").replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if verbose:
        steps.append(("4. Demojize emojis", text))

    # Remove unwanted special characters while preserving . ! ?
    text = re.sub(r'[^A-Za-z0-9\s.!?]', '', text)
    if verbose:
        steps.append(("5. Remove special characters", text))

    # Normalize newline characters
    text = re.sub(r'[\r\n]+', ' ', text)
    if verbose:
        steps.append(("6. Normalize newlines", text))

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    if verbose:
        steps.append(("7. Normalize spaces", text))

    # Normalize repeated punctuation
    text = re.sub(r'([.!?])\1+', r'\1', text)
    if verbose:
        steps.append(("8. Normalize punctuation", text))

    return text 

In [54]:
INPUT_DIR = Path(
    r"C:\Users\AadharJain\Desktop\IBM_Training-main\Mock_Test\customer_feedback"
)
products = {
    0: "cinemax", 
    1: "ecoscreen", 
    2: "smartvision", 
    3: "technova", 
    4: "ultraview", 
    5: "visionmax" 
}

raw_feedbacks = []
for i in range(len(products)):
    file_path = INPUT_DIR / f"{products[i]}.txt"
    raw_feedbacks.append(file_path.read_text(encoding="utf-8"))


In [55]:
# cleaning each raw feedaback iteratively
clean_feedbacks = []
for i in range(len(products)):
    cleaned = clean_text(raw_feedbacks[i])
    clean_feedbacks.append(cleaned)

**Save cleaned data**

In [56]:
# Create output folder inside Machine Learning folder
OUTPUT_DIR = Path(
    r"C:\Users\AadharJain\Desktop\IBM_Training-main\Mock_Test\cleaned_feedback"
)

# Create folder if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

# writing the cleaned feedbacks
for i in range(len(products)):
    output_path = OUTPUT_DIR / f"{products[i]}_cleaned.txt"
    output_path.write_text(clean_feedbacks[i], encoding="utf-8")


**Sentence chunking**

In [57]:
def sentence_chunking(text: str, sentences_per_chunk, overlap_sentences):
    sentences = sent_tokenize(text)

    chunks = []
    step = max(1, sentences_per_chunk - overlap_sentences)

    for i in range(0, len(sentences), step):
        chunks_sents = sentences[i : i + sentences_per_chunk]
        if chunks_sents:
            chunks.append({
                'id': len(chunks),
                'text': ' '.join(chunks_sents),
                'sentence_range': (i, i + len(chunks_sents)-1),
                'size': len(' '.join(chunks_sents)),
                'num_sentences': len(chunks_sents)
            })

    return chunks

In [58]:
product_chunks = []
for i in range(len(products)):
    chunks = sentence_chunking(clean_feedbacks[i], 3, 1)
    product_chunks.append(chunks)

In [69]:
product_chunks[0]

[{'id': 0,
  'text': 'Customer Feedback CineMax Ultra 55inch Smart TV The CineMax Ultra 55inch Smart TV has received mostly positive reviews from customers. smiling face with smiling eyes The picture quality is excellent and the television produces deep blacks and vibrant colors. thumbs up thumbs up Customers who watch movies frequently were particularly impressed with the cinematic viewing experience.',
  'sentence_range': (0, 2),
  'size': 383,
  'num_sentences': 3},
 {'id': 1,
  'text': 'thumbs up thumbs up Customers who watch movies frequently were particularly impressed with the cinematic viewing experience. The television also supports Dolby Vision and HDR content. The design is modern and attractive although some customers felt that the stand was not very stable.',
  'sentence_range': (2, 4),
  'size': 284,
  'num_sentences': 3},
 {'id': 2,
  'text': 'The design is modern and attractive although some customers felt that the stand was not very stable. One customer reported that t

**Creating a dataframe for products and respective chunks**

In [60]:
rows = []

for product_idx, product_name in products.items():
    for chunk in product_chunks[product_idx]:
        row = {
            "product_id": product_idx,
            "product_name": product_name
        }
        row.update(chunk)
        rows.append(row)

df = pd.DataFrame(rows)

In [61]:
df = df.drop(columns=['sentence_range'])
df.rename(columns={"id": "chunk_id"}, inplace=True)
df.head()

,product_id,product_name,chunk_id,text,size,num_sentences
0,0,cinemax,0,Customer Feedback CineMax Ultra 55inch Smart T...,383,3
1,0,cinemax,1,thumbs up thumbs up Customers who watch movies...,284,3
2,0,cinemax,2,The design is modern and attractive although s...,274,3
3,0,cinemax,3,confused face The sound quality is very good f...,244,3
4,0,cinemax,4,Streaming applications work smoothly and the o...,190,3


In [110]:
df.shape

(75, 11)

**Sentiment analysis**

In [62]:
sid = SentimentIntensityAnalyzer()
def analyze_sentiment(text):

    # Get sentiment scores
    scores = sid.polarity_scores(text)

    # Extract compound score
    compound_score = scores["compound"]

    # Classify sentiment
    if compound_score >= 0.05:
        sentiment = "Positive"

    elif compound_score <= -0.05:
        sentiment = "Negative"

    else:
        sentiment = "Neutral"

    return pd.Series({
        "Positive_Score": scores["pos"],
        "Negative_Score": scores["neg"],
        "Neutral_Score": scores["neu"],
        "Compound_Score": compound_score,
        "Sentiment": sentiment
    })

In [63]:
# APPLY SENTIMENT ANALYSIS

sentiment_results = df["text"].apply(
    analyze_sentiment
)


# Add sentiment results to original DataFrame
df = pd.concat(
    [df, sentiment_results],
    axis=1
)

In [64]:
df.head(3)

,product_id,product_name,chunk_id,text,size,num_sentences,Positive_Score,Negative_Score,Neutral_Score,Compound_Score,Sentiment
0,0,cinemax,0,Customer Feedback CineMax Ultra 55inch Smart T...,383,3,0.347,0.000,0.653,0.9764,Positive
1,0,cinemax,1,thumbs up thumbs up Customers who watch movies...,284,3,0.212,0.041,0.747,0.8266,Positive
2,0,cinemax,2,The design is modern and attractive although s...,274,3,0.120,0.134,0.747,0.0742,Positive


**Product Summary**

In [66]:
df.columns

Index(['product_id', 'product_name', 'chunk_id', 'text', 'size',
       'num_sentences', 'Positive_Score', 'Negative_Score', 'Neutral_Score',
       'Compound_Score', 'Sentiment'],
      dtype='str')

In [ ]:
summary_df = (
    df.groupby(["product_id", "product_name"])
      .agg(
          Total_Chunks=("chunk_id", "count"),
          Positive_Chunks=("Sentiment", lambda x: (x == "Positive").sum()),
          Negative_Chunks=("Sentiment", lambda x: (x == "Negative").sum()),
          Neutral_Chunks=("Sentiment", lambda x: (x == "Neutral").sum()),
          Average_Sentiment_Score=("Compound_Score", "mean")
      )
      .reset_index()
)

# Calculate percentages
summary_df["Positive_Percentage"] = (
    summary_df["Positive_Chunks"] / summary_df["Total_Chunks"] * 100
).round(2)

summary_df["Negative_Percentage"] = (
    summary_df["Negative_Chunks"] / summary_df["Total_Chunks"] * 100
).round(2)

summary_df["Neutral_Percentage"] = (
    summary_df["Neutral_Chunks"] / summary_df["Total_Chunks"] * 100
).round(2)

# Round average sentiment score
summary_df["Average_Sentiment_Score"] = (
    summary_df["Average_Sentiment_Score"].round(4)
)


In [68]:
summary_df.head()

,product_id,product_name,Total_Chunks,Positive_Chunks,Negative_Chunks,Neutral_Chunks,Average_Sentiment_Score,Positive_Percentage,Negative_Percentage,Neutral_Percentage
0,0,cinemax,12,9,1,2,0.4847,75.00,8.33,16.67
1,1,ecoscreen,12,9,2,1,0.3082,75.00,16.67,8.33
2,2,smartvision,12,9,2,1,0.4886,75.00,16.67,8.33
3,3,technova,15,11,4,0,0.3959,73.33,26.67,0.00
4,4,ultraview,12,10,1,1,0.6313,83.33,8.33,8.33


**Converting to csv**

In [70]:
summary_df.to_csv("product_sentiment_summary.csv", index=False)

**Task 12 – Identify the Best Product**

In [ ]:
summary_df.sort_values(
    by=[
        "Average_Sentiment_Score",
        "Positive_Percentage",
        "Negative_Percentage"
    ],
    ascending=[False, False, True]
)
# top one is the best

,product_id,product_name,Total_Chunks,Positive_Chunks,Negative_Chunks,Neutral_Chunks,Average_Sentiment_Score,Positive_Percentage,Negative_Percentage,Neutral_Percentage
4,4,ultraview,12,10,1,1,0.6313,83.33,8.33,8.33
2,2,smartvision,12,9,2,1,0.4886,75.00,16.67,8.33
0,0,cinemax,12,9,1,2,0.4847,75.00,8.33,16.67
3,3,technova,15,11,4,0,0.3959,73.33,26.67,0.00
5,5,visionmax,12,8,2,2,0.3225,66.67,16.67,16.67
1,1,ecoscreen,12,9,2,1,0.3082,75.00,16.67,8.33


In [ ]:
print('*'*100)

# Question 1 Which product received the most positive customer feedback?
print("Most positive customer feedback: ", end="")
most_positive = summary_df[summary_df['Positive_Percentage'] == summary_df['Positive_Percentage'].max()]
print(most_positive["product_name"].to_string(index=False))


# Question 2 Which product received the most negative feedback?
print("Most negative customer feedback: ", end="")
most_negative = summary_df[summary_df['Negative_Percentage'] == summary_df['Negative_Percentage'].max()]
print(most_negative["product_name"].to_string(index=False))

# Question 3 Which product has the highest average sentiment score?
print("highest average sentiment score: ", end="")
Average_Sentiment_Score_highest = summary_df[summary_df['Average_Sentiment_Score'] == summary_df['Average_Sentiment_Score'].max()]
print(Average_Sentiment_Score_highest["product_name"].to_string(index=False))

# Question 4 Which product has the lowest average sentiment score?
print("lowest average sentiment score: ", end="")
Average_Sentiment_Score_lowest = summary_df[summary_df['Average_Sentiment_Score'] == summary_df['Average_Sentiment_Score'].min()]
print(Average_Sentiment_Score_lowest["product_name"].to_string(index=False))



****************************************************************************************************
Most positive customer feedback: ultraview
Most negative customer feedback: technova
highest average sentiment score: ultraview
lowest average sentiment score: ecoscreen


In [111]:
paragraphs = []

# getting chunks out into the paragraphs
for feedbacks in clean_feedbacks:
    paragraphs.append( feedbacks )
# user_query = input("Enter a query to look up: ")

# Question 5 Which product is best for:
# Movie lovers?
queries = ["Which is best for movie lovers?", 
           "Which is best for gaming?",
           "Which is most budget friendly?",
           "Which is best for families?",
           "Which is premium?"]

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')

for q in queries:

    tfidf_matrix = tfidf_vectorizer.fit_transform(paragraphs + [q])

    query_tfidf_vector = tfidf_matrix[-1]
    paragraph_tfidf_vector = tfidf_matrix[:-1]

    from sklearn.metrics.pairwise import cosine_similarity
    tfidf_similarity = cosine_similarity(query_tfidf_vector, paragraph_tfidf_vector)
    tfidf_similarity.shape
    ls = list(tfidf_similarity[0])

    print(q, "",  products[ls.index(max(ls))])

Which is best for movie lovers?  cinemax
Which is best for gaming?  cinemax
Which is most budget friendly?  technova
Which is best for families?  smartvision
Which is premium?  ecoscreen


In [91]:
# Question 6 Which product would you recommend overall?
summary_df.sort_values(
    by=[
        "Average_Sentiment_Score",
        "Positive_Percentage",
        "Negative_Percentage"
    ],
    ascending=[False, False, True]
)

,product_id,product_name,Total_Chunks,Positive_Chunks,Negative_Chunks,Neutral_Chunks,Average_Sentiment_Score,Positive_Percentage,Negative_Percentage,Neutral_Percentage
4,4,ultraview,12,10,1,1,0.6313,83.33,8.33,8.33
2,2,smartvision,12,9,2,1,0.4886,75.00,16.67,8.33
0,0,cinemax,12,9,1,2,0.4847,75.00,8.33,16.67
3,3,technova,15,11,4,0,0.3959,73.33,26.67,0.00
5,5,visionmax,12,8,2,2,0.3225,66.67,16.67,16.67
1,1,ecoscreen,12,9,2,1,0.3082,75.00,16.67,8.33
